# ptxray on a free Colab GPU

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

This notebook builds ptxray, runs the full test suite on a real NVIDIA GPU, and produces the three result tables used in the README:
1. GPU test suite (interposition, fatbinary parsing, image selection, diagnostics)
2. Tracer overhead per kernel launch
3. PTX JIT experiment: native SASS vs PTX-only, JIT cache, lazy vs eager module loading

All results are also saved to `results/` and downloaded at the end.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!nvcc --version | tail -2

In [ ]:
# Replace with your GitHub username after you push the repo
GITHUB_USER = "Adityaarvind19"
%cd /content
!rm -rf /content/ptxray && git clone -q https://github.com/{GITHUB_USER}/ptxray.git /content/ptxray
%cd /content/ptxray
!mkdir -p results

In [ ]:
%cd /content/ptxray
# Build the tracer, the GPU tests and the benchmarks for this GPU's architecture
!make -j4 all gpu 2>&1 | tail -5

In [ ]:
%cd /content/ptxray
# 1a. CPU unit tests + interposition smoke test
!make test smoke 2>&1 | tail -3

In [ ]:
%cd /content/ptxray
# 1b. GPU test suite
!bash tests/run_gpu_tests.sh 2>&1 | tee results/gpu_tests.txt

In [ ]:
%cd /content/ptxray
# What the tracer report looks like (the basic test)
!cat build/gpu-test-out/basic.txt

In [ ]:
%cd /content/ptxray
# 2. Tracer overhead per kernel launch
!bash bench/run_bench.sh 100000 2>&1 | tee results/overhead.md

In [ ]:
%cd /content/ptxray
# 3. PTX JIT experiment
!bash bench/run_jit.sh 2>&1 | tee results/jit.md

In [ ]:
%cd /content/ptxray
# Full report for the PTX-only binary (fatbinary inventory, image selection, first-launch diagnostics)
!CUDA_CACHE_DISABLE=1 ./bin/ptxray -o results/jit_ptx.json -r results/jit_ptx_report.txt --dump-ptx results/ptx -- ./build/jit_ptx
!cat results/jit_ptx_report.txt
!ls -la results/ptx && head -20 results/ptx/*.ptx

## Optional: a real framework (PyTorch)
The naive loop copies from pageable memory and calls `.item()` every step; `--fixed` uses pinned memory and logs every 50 steps. Compare the two reports.

In [ ]:
%cd /content/ptxray
!./bin/ptxray -o results/pytorch_naive.json -r results/pytorch_naive.txt -- python examples/pytorch_train.py
!./bin/ptxray -o results/pytorch_fixed.json -r results/pytorch_fixed.txt -- python examples/pytorch_train.py --fixed
!head -45 results/pytorch_naive.txt
!grep -E "Diagnostics|\\[" results/pytorch_fixed.txt

## Save results
Open any `.json` trace at https://ui.perfetto.dev (drag and drop) and take a screenshot for the README.

In [ ]:
%cd /content/ptxray
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")